In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path

def load_environment_variables():
    """
    Loads the environment variables from .env.
    Loads OpenAI api key and target model name used in the experiment
    New environment variables as necessary will be appended
    """

    PROJECT_ROOT = Path.cwd().parent
    ENV_FILE = PROJECT_ROOT / ".env"

    if ENV_FILE.exists():
        print(
            f".env file located at: {ENV_FILE.resolve()}")
        
        load_dotenv()
        env = {
                "api_key":              os.getenv("OPENAI_API_KEY"),
                "model_name":           os.getenv("OPENAI_MODEL_NAME"),
                "input_token_price":    float(os.getenv('MODEL_INPUT_TOKEN_PRICE')),
                "output_token_price":   float(os.getenv('MODEL_OUTPUT_TOKEN_PRICE'))
            }

    else:
        print(
            f".env not found at: {ENV_FILE.resolve()}"
        )
        env = {
            "api_key":              None,
            "model_name":           None,
            "input_token_price":    None,
            "output_token_price":   None
        }

    return env

In [2]:
from openai import OpenAI

env = load_environment_variables()

client = OpenAI(
    api_key=env['api_key']
)

response = client.chat.completions.create(
    model=env['model_name'],
    messages=[
        {
            "role": "user",
            "content": "what is RAG?"
        }
    ]
)

print(response.choices[0].message.content)

.env file located at: D:\Bits-lab\eduassist-llmops-prompt-evaluation\.env
RAG can refer to a few different things depending on the context:

1. **RAG Status**: In project management, RAG refers to a color-coding system that indicates the status of a project or task. RAG stands for Red, Amber (or Yellow), and Green. 
   - **Red** indicates significant issues or risks that need immediate attention.
   - **Amber** indicates that there are some concerns that could affect the project but are manageable.
   - **Green** indicates that the project is on track and progressing well.

2. **Retrieval-Augmented Generation (RAG)**: In the context of machine learning and natural language processing, RAG is a model architecture that combines retrieval and generation. It allows a model to retrieve relevant documents from a database and then generate text based on those documents to produce more informed and contextually relevant outputs. This approach is useful in applications like question answering a

In [3]:
response = client.chat.completions.create(
    model=env['model_name'],
    messages=[
        {
            "role": "user",
            "content": "what is RAG in Natural Language Processing?"
        }
    ]
)

print(response.choices[0].message.content)

RAG, or Retrieval-Augmented Generation, is a model architecture in Natural Language Processing that combines generative models with retrieval-based techniques to enhance the generation of text. It was introduced by researchers at Facebook AI Research (FAIR) and published in a paper titled "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks."

### Key Components of RAG:

1. **Retrieval Component:**
   - The retrieval component of RAG is responsible for fetching relevant documents or pieces of information from a large external knowledge base or corpus. This is typically done using a retriever model, which can be based on dense or sparse retrieval methods.
   - The goal is to gather contextually relevant information that can aid in generating more accurate and informative responses.

2. **Generation Component:**
   - After retrieving relevant documents, the generative model (often based on architectures like BERT or GPT) takes the retrieved information as additional context 

In [4]:
questions = [
    "what is overfitting in machine learning?",
    "Explain gradient descent in simple terms.",
    "what is neural network?",
    "Difference between supervised and unsupervised learning?",
    "Explain backpropagation in neural networks."
]

In [5]:
PROMT_V1 = """
Answer the following student questions clearly.
Question: {Question}
"""

In [6]:
import time
import pandas as pd

def evaluate_prompt(
        questions,
        prompt_template,
        prompt_version,
        model,
        input_token_price,
        output_token_price
):
    results = []

    for idx, question in enumerate(questions, start=1):

        prompt = prompt_template.format(
            Question=question
        )
        
        start_time = time.time()

        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        end_time = time.time()

        latency = end_time - start_time

        # token cost calculation
        input_token_price = env['input_token_price']
        output_token_price = env['output_token_price']

        input_cost = (
            response.usage.prompt_tokens / (1024 * 1024) 
        ) * input_token_price

        output_cost = (
            response.usage.completion_tokens / (1024 * 1024)
        ) * output_token_price

        total_cost = input_cost + output_cost
        
        results.append(
            {
                "prompt": prompt,
                "response": response.choices[0].message.content,
                "question_id": idx,
                "prompt_version": prompt_version,
                "model": "gpt-4.1-mini",
                "latency_seconds": latency,
                "input_tokens": response.usage.prompt_tokens,
                "output_tokens": response.usage.completion_tokens,
                "total_tokens": response.usage.total_tokens,
                "input_token_price": input_cost,
                "output_token_price": output_cost,
                "total_cost": total_cost,
                "response_status": "success"


            }
        )

    return pd.DataFrame(results)

In [7]:
df_v1 = evaluate_prompt(
    questions,
    PROMT_V1,
    "v1",
    env['model_name'],
    env['input_token_price'],
    env['output_token_price']
)

print("Statistics of V1 evaluation")
print(df_v1['latency_seconds'].describe())

latency_stats_v1 = df_v1['latency_seconds'].agg(
    ['mean', 'median', "min", 'max', 'std']
).to_frame("value")

print("Selected parameters")
print(latency_stats_v1)

for _, row in df_v1.iterrows():
    print("PROMPT:")
    print(row["prompt"])

    print("\nRESPONSE:")
    print(row["response"])

    print("\n" + "="*80 + "\n")

Statistics of V1 evaluation
count    5.000000
mean     3.458624
std      1.442517
min      2.218092
25%      2.356238
50%      2.678500
75%      4.815259
max      5.225033
Name: latency_seconds, dtype: float64
Selected parameters
           value
mean    3.458624
median  2.678500
min     2.218092
max     5.225033
std     1.442517
PROMPT:

Answer the following student questions clearly.
Question: what is overfitting in machine learning?


RESPONSE:
Overfitting in machine learning refers to a modeling error that occurs when a model learns the training data too well, capturing noise and outliers instead of the underlying patterns. As a result, the model performs exceptionally well on the training dataset but fails to generalize to new, unseen data.

In simpler terms, an overfitted model is like a student who memorizes answers for a test instead of understanding the concepts. When faced with different questions (or data), the memorization doesn't help, leading to poor performance.

To avoi

In [8]:
def add_cost_metrics(df, env):
    """
    Function to calculate and add cost metric to input data frame.
    The method assumes that the dataframe contains input and output token price.
    """
    input_token_price = env['input_token_price']
    output_token_price = env['output_token_price']

    df['input_cost'] = (
        df['input_tokens'] / (1024 * 1024) 
    ) * input_token_price

    df['output_cost'] = (
        df['output_tokens'] / (1024 * 1024)
    ) * output_token_price

    df['total_cost'] = df['input_cost'] + df['output_cost']

    return df

In [9]:
#df_v1 = add_cost_metrics(df=df_v1, env=env)

df_v1.info()

<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   prompt              5 non-null      str    
 1   response            5 non-null      str    
 2   question_id         5 non-null      int64  
 3   prompt_version      5 non-null      str    
 4   model               5 non-null      str    
 5   latency_seconds     5 non-null      float64
 6   input_tokens        5 non-null      int64  
 7   output_tokens       5 non-null      int64  
 8   total_tokens        5 non-null      int64  
 9   input_token_price   5 non-null      float64
 10  output_token_price  5 non-null      float64
 11  total_cost          5 non-null      float64
 12  response_status     5 non-null      str    
dtypes: float64(4), int64(4), str(5)
memory usage: 7.5 KB


In [10]:
# Average monthly cost
avg_cost = df_v1['total_cost'].mean()

print(f"Average cost per question: ${avg_cost:.4f}")

projected_monthly_cost = avg_cost * 10000

print(f"Project monthly average cost: ${projected_monthly_cost:.2f}", )

Average cost per question: $0.0001
Project monthly average cost: $1.48


In [11]:
# Promt V2
PROMT_V2 = """
You are an academic assistant.
Answer the following student question in a clear, concise and accurate manner.
Use simple language suitable for undergraduate students.
Question: {Question}
"""

In [12]:
df_v2 = evaluate_prompt(
    questions,
    PROMT_V2,
    "v2",
    env['model_name'],
    env['input_token_price'],
    env['output_token_price']
)

print("Statistics of V2 evaluation")
print(df_v2['latency_seconds'].describe())

latency_stats_v2 = df_v2['latency_seconds'].agg(
    ['mean', 'median', "min", 'max', 'std']
).to_frame("value")

print("Selected parameters")
print(latency_stats_v2)

for _, row in df_v2.iterrows():
    print("PROMPT:")
    print(row["prompt"])

    print("\nRESPONSE:")
    print(row["response"])

    print("\n" + "="*80 + "\n")
# Average monthly cost
avg_cost_v2 = df_v2['total_cost'].mean()

print(f"Average cost per question: ${avg_cost_v2:.4f}")

projected_monthly_cost_v2 = avg_cost_v2 * 10000

print(f"Project monthly average cost: ${projected_monthly_cost_v2:.2f}", )

Statistics of V2 evaluation
count    5.000000
mean     2.764752
std      0.795662
min      1.847187
25%      2.205728
50%      2.629820
75%      3.395623
max      3.745401
Name: latency_seconds, dtype: float64
Selected parameters
           value
mean    2.764752
median  2.629820
min     1.847187
max     3.745401
std     0.795662
PROMPT:

You are an academic assistant.
Answer the following student question in a clear, concise and accurate manner.
Use simple language suitable for undergraduate students.
Question: what is overfitting in machine learning?


RESPONSE:
Overfitting in machine learning is when a model learns the training data too well, including its noise and outliers. Instead of finding the underlying patterns, the model becomes too complex and fits the training data very closely. As a result, it performs well on the training data but poorly on new, unseen data. 

Think of it like memorizing answers for a test instead of understanding the material. You might do great on the 

In [13]:
PROMPT_V3 = """
You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Use this exact JSON schema:
{{
    "answer": "string",
    "confidence": "high|medium|low",
    "key_points": ["string"]
}}

Question" {Question}
"""

In [14]:
df_v3 = evaluate_prompt(
    questions,
    PROMPT_V3,
    "v3",
    env['model_name'],
    env['input_token_price'],
    env['output_token_price']
)

print("Statistics of V3 evaluation")
print(df_v3['latency_seconds'].describe())

latency_stats_v3 = df_v3['latency_seconds'].agg(
    ['mean', 'median', "min", 'max', 'std']
).to_frame("value")

print("Selected parameters")
print(latency_stats_v3)

for _, row in df_v3.iterrows():
    print("PROMPT:")
    print(row["prompt"])

    print("\nRESPONSE:")
    print(row["response"])

    print("\n" + "="*80 + "\n")
# Average monthly cost
avg_cost_v3 = df_v3['total_cost'].mean()

print(f"Average cost per question: ${avg_cost_v3:.4f}")

projected_monthly_cost_v3 = avg_cost_v3 * 10000

print(f"Project monthly average cost: ${projected_monthly_cost_v3:.2f}", )

Statistics of V3 evaluation
count    5.000000
mean     2.104770
std      0.473271
min      1.654129
25%      1.912454
50%      1.978850
75%      2.074216
max      2.904200
Name: latency_seconds, dtype: float64
Selected parameters
           value
mean    2.104770
median  1.978850
min     1.654129
max     2.904200
std     0.473271
PROMPT:

You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Use this exact JSON schema:
{
    "answer": "string",
    "confidence": "high|medium|low",
    "key_points": ["string"]
}

Question" what is overfitting in machine learning?


RESPONSE:
{
    "answer": "Overfitting in machine learning occurs when a model learns the noise and details of the training data to the extent that it negatively impacts the model's performance on new data. Essentially, the model becomes too complex and specific to the training dataset, failing to generalize well to unseen examples.",
    "confidence": "high",
    "key_points": [
        "Over

In [15]:
# JSON validation
import json

def is_valid_json(text):
    try:
        json.loads(text)
        return 1
    except:
        return 0


In [16]:
df_v3["valid_json"] = (
    df_v3["response"].apply(is_valid_json)
)

valid_json_rate = df_v3["valid_json"].mean()
print(valid_json_rate)

1.0


In [17]:
# Prompt performance comparison

comparison_df = pd.DataFrame(
    [
        {
            "Prompt Version": "v1",
            "Avg Latency": df_v1["latency_seconds"].mean(),
            "Avg Input Tokens": df_v1["input_tokens"].mean(),
            "Avg Ouput Tokens": df_v1["output_tokens"].mean(),
            "Avg Total Tokens": df_v1["total_tokens"].mean(),
            "Avg Cost": df_v1["total_cost"].mean(),
            "JSON Valid": 0
        },
        {
            "Prompt Version": "v2",
            "Avg Latency": df_v2["latency_seconds"].mean(),
            "Avg Input Tokens": df_v2["input_tokens"].mean(),
            "Avg Ouput Tokens": df_v2["output_tokens"].mean(),
            "Avg Total Tokens": df_v2["total_tokens"].mean(),
            "Avg Cost": df_v2["total_cost"].mean(),
            "JSON Valid": 0
        },
        {
            "Prompt Version": "v3",
            "Avg Latency": df_v3["latency_seconds"].mean(),
            "Avg Input Tokens": df_v3["input_tokens"].mean(),
            "Avg Ouput Tokens": df_v3["output_tokens"].mean(),
            "Avg Total Tokens": df_v3["total_tokens"].mean(),
            "Avg Cost": df_v3["total_cost"].mean(),
            "JSON Valid": df_v3["valid_json"].mean()
        }
    ]
)

comparison_df

,Prompt Version,Avg Latency,Avg Input Tokens,Avg Ouput Tokens,Avg Total Tokens,Avg Cost,JSON Valid
0,v1,3.458624,24.6,252.2,276.8,0.000148,0.0
1,v2,2.764752,45.6,205.2,250.8,0.000124,0.0
2,v3,2.104770,67.6,138.8,206.4,0.000089,1.0


### Comparison of performance for three types of prompting

The comparison of performance for three types of prompting can be summarized as follows:
* **V1**: This version uses a basic prompt structure that provides minimal context. It is effective for straightforward tasks but may struggle with complex queries.
* **V2**: This version incorporates additional context. This instructs Language Model to assume an Academic Assistant's role.
It tends to perform better on tasks that require understanding nuanced instructions or generating more detailed responses.
* **V3**: This version follows **V2** but enhances further in terms of response format. The prompt expects the result in a JSON format, which is particularly useful for structured data extraction and integration with other systems. It is the most effective for tasks that require precise output formatting and data handling.

### Performance Metrics
* **Token Consumption**: **V1** Consumes the highest number of tokens considering the input context is minimal. The language model continues to generate more response and therefore total tokens are higher. **V1** has lesser input tokens in comparison to **V2** and **V3**, howeever the output tokens are higher.
**V3** has the total least number of tokens due its precise context and clear output structure. Following the output structure, language model generates only relevant information. Hence total tokens are lesser in comparison to **V1** and **V2**.
* **Latency**: Latency is low **V3** in comparison to **V1** and **V2**. The structured output format allows for faster processing and integration with other systems, reducing the time taken to parse and utilize the generated data.
* **Cost**: **V3** has the least cost for the given combination of questions and generated answers. The output tokens generated in **V3** are less in comparison to other prompts due to a well defined structure. This overall leads to lesser token price overall.

In [44]:
all_results = pd.concat(
    [df_v1, df_v2, df_v3],
    ignore_index=True
)

print(all_results["prompt_version"].value_counts())

all_results.to_csv(
    "../outputs/latency_tokens_cost.csv",
    index=False
)

prompt_version
v1    5
v2    5
v3    5
Name: count, dtype: int64


In [45]:
comparison_df.to_csv(
    "../outputs/prompt_evaluation_results.csv",
    index=False
)

In [18]:
import mlflow

PROJECT_PATH = Path.cwd().parent

mlflow.set_tracking_uri(
    f"sqlite:///{PROJECT_PATH}/mlflow.db"
)

mlflow.set_experiment(
    "eduassist_llmops_assignment"
)

<Experiment: artifact_location='file:///d:/Bits-lab/eduassist-llmops-prompt-evaluation/notebooks/mlruns/1', creation_time=1793676484295, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1793676484295, lifecycle_stage='active', name='eduassist_llmops_assignment', tags={}, trace_location=None, workspace='default'>

In [19]:
def log_prompt_metrics(
        prompt_version,
        df,
        valid_json=0
):
    with mlflow.start_run(run_name=prompt_version):
        mlflow.log_param("prompt_version", prompt_version)
        mlflow.log_metric("avg_latency_seconds", df['latency_seconds'].mean())
        mlflow.log_metric("avg_input_tokens", df["input_tokens"].mean())
        mlflow.log_metric("avg_output_tokens", df["output_tokens"].mean())
        mlflow.log_metric("avg_cost_per_request", df["total_cost"].mean())
        mlflow.log_metric("valid_json", valid_json)


In [20]:
log_prompt_metrics(prompt_version="v1", df=df_v1)
log_prompt_metrics(prompt_version="v2", df=df_v2)
log_prompt_metrics(prompt_version="v3", df=df_v3, valid_json=valid_json_rate)

In [21]:
print(mlflow.__version__)
print(dir(mlflow))

print(hasattr(mlflow, "genai"))

3.17.2
['ActiveModel', 'ActiveRun', 'IS_TRACING_SDK_ONLY', 'Image', 'LazyLoader', 'MLFLOW_CONFIGURE_LOGGING', 'MlflowClient', 'MlflowException', 'RunOperations', 'TYPE_CHECKING', 'VERSION', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_configure_mlflow_loggers', '_install_sensitive_query_param_filter', 'active_run', 'add_trace', 'ag2', 'agent', 'agno', 'anthropic', 'artifacts', 'autogen', 'autolog', 'azure', 'bedrock', 'catboost', 'clear_active_model', 'client', 'config', 'context', 'contextlib', 'create_experiment', 'create_external_model', 'create_workspace', 'crewai', 'data', 'delete_assessment', 'delete_experiment', 'delete_experiment_tag', 'delete_logged_model_tag', 'delete_prompt_alias', 'delete_run', 'delete_tag', 'delete_trace_tag', 'delete_workspace', 'diffusers', 'disable_system_metrics_logging', 'doctor', 'dspy', 'enable_system_metrics_logging', 'end_run', 'entities', 'environ

In [22]:
print(dir(mlflow.genai))

['Agent', 'ConversationSimulator', 'EvaluationDatasetVersion', 'LabelingSession', 'ReviewApp', 'Scorer', 'ScorerScheduleConfig', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'agent_tester', 'create_dataset', 'create_labeling_session', 'create_mcp_access_endpoint', 'datasets', 'delete_dataset', 'delete_dataset_tag', 'delete_labeling_session', 'delete_mcp_access_endpoint', 'delete_mcp_server', 'delete_mcp_server_alias', 'delete_mcp_server_tag', 'delete_mcp_server_version', 'delete_mcp_server_version_tag', 'delete_prompt_alias', 'delete_prompt_model_config', 'delete_prompt_tag', 'delete_prompt_version_tag', 'delete_scorer', 'disable_git_model_versioning', 'discovery', 'enable_git_model_versioning', 'evaluate', 'evaluation', 'get_dataset', 'get_labeling_session', 'get_labeling_sessions', 'get_latest_mcp_server_version', 'get_mcp_access_endpoint', 'get_mcp_server', 'get_mcp_server_version', 'get_mcp_server_v

In [23]:
for item in dir(mlflow.genai):
    print(item)

Agent
ConversationSimulator
EvaluationDatasetVersion
LabelingSession
ReviewApp
Scorer
ScorerScheduleConfig
__all__
__builtins__
__cached__
__doc__
__file__
__loader__
__name__
__package__
__path__
__spec__
agent_tester
create_dataset
create_labeling_session
create_mcp_access_endpoint
datasets
delete_dataset
delete_dataset_tag
delete_labeling_session
delete_mcp_access_endpoint
delete_mcp_server
delete_mcp_server_alias
delete_mcp_server_tag
delete_mcp_server_version
delete_mcp_server_version_tag
delete_prompt_alias
delete_prompt_model_config
delete_prompt_tag
delete_prompt_version_tag
delete_scorer
disable_git_model_versioning
discovery
enable_git_model_versioning
evaluate
evaluation
get_dataset
get_labeling_session
get_labeling_sessions
get_latest_mcp_server_version
get_mcp_access_endpoint
get_mcp_server
get_mcp_server_version
get_mcp_server_version_by_alias
get_prompt_tags
get_review_app
get_scorer
git_versioning
judges
label_schemas
labeling
list_scorers
load_prompt
make_judge
make_sc

In [ ]:
# prompt registry for PROMPT_V1
prompt_v1 = mlflow.genai.register_prompt(
    name="eduassist_qa_prompt",
    template=PROMT_V1,
    commit_message="Initial baseline prompt"
)

print(prompt_v1)

PromptVersion(name=eduassist_qa_prompt, version=1, template=
Answer the following student ...)


In [ ]:
# prompt registry for PROMPT_V2
prompt_v2 = mlflow.genai.register_prompt(
    name="eduassist_qa_prompt",
    template=PROMT_V2,
    commit_message="Improved academic assistant instructions"
)

print(prompt_v2)

PromptVersion(name=eduassist_qa_prompt, version=2, template=
You are an academic assistant...)


In [ ]:
# prompt registry for PROMPT_V3
prompt_v3 = mlflow.genai.register_prompt(
    name="eduassist_qa_prompt",
    template=PROMPT_V3,
    commit_message="Added Structured JSON output requirement"
)

print(prompt_v3)

PromptVersion(name=eduassist_qa_prompt, version=3, template=
You are an academic assistant...)


In [ ]:
#CONFIRMATION of prompt registry
for i in range(0,3):
    print(i)
    p = mlflow.genai.load_prompt(
        f"prompts:/eduassist_qa_prompt/{i+1}"
    )
    print(p)

0
PromptVersion(name=eduassist_qa_prompt, version=1, template=
Answer the following student ...)
1
PromptVersion(name=eduassist_qa_prompt, version=2, template=
You are an academic assistant...)
2
PromptVersion(name=eduassist_qa_prompt, version=3, template=
You are an academic assistant...)


In [34]:
# Adding project tag
mlflow.genai.set_prompt_tag(
    "eduassist_qa_prompt",
    "project",
    "eduassist-llmops"
)

In [35]:
# Promoting the best prompt version
# From the token, cost and latency analysis, PROMPT_V3 is the winner in this experiment.
# Hence PROMPT_V3 will be promoted to Production

mlflow.genai.set_prompt_alias(
    name="eduassist_qa_prompt",
    alias="production",
    version=3 #PROMPT_V3 is registered as 3rd version
)

In [36]:
# Verification of production promotion
production_prompt = mlflow.genai.load_prompt(
    "prompts:/eduassist_qa_prompt@production"
)

print(production_prompt)

PromptVersion(name=eduassist_qa_prompt, version=3, template=
You are an academic assistant...)


In [41]:
# Prompt comparison table

import pandas as pd

prompt_comparison = pd.DataFrame({
    "Version": ["V1", "V2", "V3"],
    "Prompt": [PROMT_V1, PROMT_V2, PROMPT_V3]
})

from IPython.display import Markdown, display

for _, row in prompt_comparison.iterrows():
    display(Markdown(f"## {row['Version']}"))
    display(Markdown(f"```text\n{row['Prompt']}\n```"))


## V1

```text

Answer the following student questions clearly.
Question: {Question}

```

## V2

```text

You are an academic assistant.
Answer the following student question in a clear, concise and accurate manner.
Use simple language suitable for undergraduate students.
Question: {Question}

```

## V3

```text

You are an academic assistant.
Answer the student question and return ONLY valid JSON.
Use this exact JSON schema:
{{
    "answer": "string",
    "confidence": "high|medium|low",
    "key_points": ["string"]
}}

Question" {Question}

```